# Endo-FM smoke test on PitVis-2023 (1 video)

Verifies the pipeline end-to-end on `video_01.mp4` + `annotations_01.csv`
before running it for real on the full dataset. Small evenly-spaced sample,
direct-seek frame grabs (not a full sequential decode). Not a benchmark --
sample size is far too small for a real number.

## 0. Setup

In [ ]:

import sys, subprocess, random
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/endofm-benchmark")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "einops", "timm", "gdown", "opencv-python-headless"], check=True)
else:
    PROJECT_ROOT = Path(r"E:/endofm-benchmark")

PITVIS_DIR = PROJECT_ROOT / "pitvis"
VIDEO_PATH = PITVIS_DIR / "video_01.mp4"
ANNOT_PATH = PITVIS_DIR / "annotations_01.csv"
FRAMES_DIR = PITVIS_DIR / "smoketest_frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

ENDOFM_REPO = PROJECT_ROOT / "Endo-FM"
CKPT_DIR = PROJECT_ROOT / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ENDOFM_REPO))

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(0)
torch.manual_seed(0)


## 1. Backbone

In [ ]:

if not (ENDOFM_REPO / "models" / "timesformer.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/med-air/Endo-FM.git", str(ENDOFM_REPO)], check=True)

CKPT_PATH = CKPT_DIR / "endo_fm.pth"
if not CKPT_PATH.exists():
    import gdown
    gdown.download(id="1H7B91Ewm4QkZRsnUk1Bn0IQch5P8C7Xl", output=str(CKPT_PATH), quiet=False)


## 2. Load annotations

In [ ]:

import pandas as pd

df = pd.read_csv(ANNOT_PATH)
print(df.shape)
print(df["int_step"].value_counts().sort_index())


## 3. Sample rows evenly across the video

In [ ]:

N_SAMPLES = 40

sample_idx = sorted(random.Random(0).sample(range(len(df)), min(N_SAMPLES, len(df))))
sample_df = df.iloc[sample_idx].reset_index(drop=True)
print(f"Sampled {len(sample_df)} rows, int_time range {sample_df['int_time'].min()}-{sample_df['int_time'].max()}")
sample_df.head()


## 4. Extract sampled frames (direct seek, not sequential decode)

In [ ]:

import cv2
from tqdm import tqdm

def frame_path(int_time):
    return FRAMES_DIR / f"{int_time}.jpg"

cap = cv2.VideoCapture(str(VIDEO_PATH))
for int_time in tqdm(sample_df["int_time"], desc="Extracting"):
    out_path = frame_path(int_time)
    if out_path.exists():
        continue
    cap.set(cv2.CAP_PROP_POS_MSEC, int_time * 1000)
    ok, frame = cap.read()
    if not ok:
        print(f"WARNING: failed to read frame at t={int_time}s")
        continue
    cv2.imwrite(str(out_path), frame)
cap.release()

sample_df = sample_df[sample_df["int_time"].apply(lambda t: frame_path(t).exists())].reset_index(drop=True)
print(f"Usable frames: {len(sample_df)}")


## 5. Model

In [ ]:

from models.timesformer import get_vit_base_patch16_224

class SimpleCfg:
    class DATA:
        TRAIN_CROP_SIZE = 224
        NUM_FRAMES = 8
    class MODEL:
        NUM_CLASSES = 0
    class TIMESFORMER:
        ATTENTION_TYPE = "divided_space_time"
        PRETRAINED_MODEL = ""

def build_backbone(load_checkpoint: bool):
    model = get_vit_base_patch16_224(cfg=SimpleCfg(), no_head=True)
    if load_checkpoint:
        ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
        if "teacher" in ckpt:
            ckpt = ckpt["teacher"]
        state_dict = {k[len("backbone."):]: v for k, v in ckpt.items() if k.startswith("backbone.")}
        msg = model.load_state_dict(state_dict, strict=False)
        print("Loaded Endo-FM weights:", msg)
    model.eval().to(DEVICE)
    return model

NUM_FRAMES = SimpleCfg.DATA.NUM_FRAMES
IMG_SIZE = SimpleCfg.DATA.TRAIN_CROP_SIZE
model = build_backbone(load_checkpoint=True)


## 6. Data pipeline + feature extraction

In [ ]:

import numpy as np
from PIL import Image
import torchvision.transforms as T

preprocess = T.Compose([
    T.Resize(256),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

def load_clip_tensor(path):
    frame = preprocess(Image.open(path).convert("RGB"))
    return frame.unsqueeze(1).repeat(1, NUM_FRAMES, 1, 1)

@torch.no_grad()
def extract_features():
    feats = []
    for int_time in tqdm(sample_df["int_time"], desc="Features"):
        clip = load_clip_tensor(frame_path(int_time)).unsqueeze(0).to(DEVICE)
        f = model(clip)
        feats.append(f.cpu().numpy()[0])
    return np.stack(feats)

features = extract_features()
print("Feature matrix shape:", features.shape)
assert features.shape == (len(sample_df), 768), "unexpected feature shape -- pipeline is broken"
print("PASS: model loads, forward pass runs, output shape is correct.")


## 7. Trivial end-to-end check (NOT a benchmark -- sample far too small)

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from collections import Counter

labels = sample_df["int_step"].to_numpy()
label_counts = Counter(labels)
print("Sampled label distribution:", label_counts)

if len(label_counts) < 2 or min(label_counts.values()) < 2:
    print("Too few samples per class in this tiny sample to fit/split meaningfully -- "
          "that's expected at N=40 across 13 classes, not a bug. "
          "Feature extraction above (Section 6) is the actual smoke test; this cell is optional bonus.")
else:
    Xtr, Xte, ytr, yte = train_test_split(features, labels, test_size=0.3, random_state=0, stratify=labels)
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(Xtr, ytr)
    acc = clf.score(Xte, yte)
    print(f"Trivial-split accuracy: {acc:.3f} (meaningless as a number, N is tiny -- just confirms sklearn.fit/predict runs on these features)")


## Summary

In [ ]:

print("Smoke test passed:" if features.shape[1] == 768 else "Smoke test FAILED:")
print(f"  video: {VIDEO_PATH.name}")
print(f"  sampled frames: {len(sample_df)}")
print(f"  feature dim: {features.shape[1]}")
print("Ready to scale up to the full 25-video PitVis benchmark once the rest of the dataset is available.")
